# Stage 4: Merge Traditional and News Features

Load cached outputs from Stage 1-3, build lagged rolling sentiment features, and save final modeling table.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data' / 'intermediate'

factors = pd.read_parquet(DATA_DIR / 'factors_traditional.parquet')
target = pd.read_parquet(DATA_DIR / 'target_next_ret.parquet')
tone_wide = pd.read_parquet(DATA_DIR / 'gdelt_tone_wide.parquet')
finbert_wide = pd.read_parquet(DATA_DIR / 'finbert_daily_wide.parquet')
count_wide = pd.read_parquet(DATA_DIR / 'news_count_wide.parquet')

# Strip timezone so indices align with timezone-naive factors index
def to_naive(idx):
    idx = pd.to_datetime(idx, utc=True)
    return idx.tz_convert(None)

tone_wide.index = to_naive(tone_wide.index)
finbert_wide.index = to_naive(finbert_wide.index)
count_wide.index = to_naive(count_wide.index)

print('Loaded traditional factors:', factors.shape)

In [ ]:
# Reindex to full daily calendar before rolling.
# Without this, rolling(7) means "last 7 news articles" (row-based), not
# "last 7 calendar days". shift(1) also lags by 1 news event not 1 trading day.
_d0 = min(finbert_wide.index.min(), tone_wide.index.min())
_d1 = max(finbert_wide.index.max(), tone_wide.index.max())
_cal = pd.date_range(_d0, _d1, freq='D')
finbert_wide = finbert_wide.reindex(_cal)
tone_wide    = tone_wide.reindex(_cal)
count_wide   = count_wide.reindex(_cal).fillna(0)

def rolling_mean(df, w):
    return df.rolling(f'{w}D', min_periods=1).mean()

def rolling_sum(df, w):
    return df.rolling(f'{w}D', min_periods=1).sum()

def wide_to_long(df, col):
    s = df.stack()
    s.index.set_names(['date', 'ticker'], inplace=True)
    return s.to_frame(col)

fb7 = rolling_mean(finbert_wide, 7).shift(1)
fb30 = rolling_mean(finbert_wide, 30).shift(1)
gt7 = rolling_mean(tone_wide, 7).shift(1)
gt30 = rolling_mean(tone_wide, 30).shift(1)
nv7 = rolling_sum(count_wide, 7).shift(1)
nv30 = rolling_sum(count_wide, 30).shift(1)
smom = fb7 - fb30

sent_feats = (
    wide_to_long(fb7, 'finbert_sent_7d')
    .join(wide_to_long(fb30, 'finbert_sent_30d'), how='outer')
    .join(wide_to_long(smom, 'sent_momentum'), how='outer')
    .join(wide_to_long(gt7, 'gdelt_tone_7d'), how='outer')
    .join(wide_to_long(gt30, 'gdelt_tone_30d'), how='outer')
    .join(wide_to_long(nv7, 'news_volume_7d'), how='outer')
    .join(wide_to_long(nv30, 'news_volume_30d'), how='outer')
).sort_index()

model_table = factors.join(sent_feats, how='left').join(target[['ret_t1']], how='left')
model_table = model_table.sort_index()

model_table.to_parquet(DATA_DIR / 'model_table.parquet')
sent_feats.to_parquet(DATA_DIR / 'sentiment_features_long.parquet')

print('Saved model_table.parquet and sentiment_features_long.parquet')
print('Model table shape:', model_table.shape)